In [ ]:
# ── CELL 1: Imports & Configuration ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, re, time
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

FAKE_COLOR = '#e24b4a'
REAL_COLOR = '#1d9e75'
ACCENT     = '#7f77dd'

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#333355', 'axes.labelcolor': '#e0e0ff',
    'text.color': '#e0e0ff', 'xtick.color': '#e0e0ff',
    'ytick.color': '#e0e0ff', 'grid.color': '#333355',
    'grid.alpha': 0.3, 'font.family': 'monospace',
})
print('All imports successful.')

In [ ]:
# ── CELL 2: Load Dataset ──────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
# Place Fake.csv and True.csv in the same directory as this notebook
fake = pd.read_csv('Fake.csv')
real = pd.read_csv('True.csv')

fake['label'] = 0   # 0 = Fake
real['label'] = 1   # 1 = Real

df = pd.concat([fake, real], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Shape         : {df.shape}')
print(f'Fake articles : {(df.label==0).sum():,}')
print(f'Real articles : {(df.label==1).sum():,}')
print(f'Columns       : {list(df.columns)}')
print('\nNull values:')
print(df.isnull().sum())
df.head()

In [ ]:
# ── CELL 3: Feature Engineering & Text Preprocessing ─────────────────────────
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'<.*?>', '', text)             # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)          # keep letters only
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Combine title + article body for richer signal
df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Stylometric features
df['word_count']     = df['combined'].apply(lambda x: len(str(x).split()))
df['char_count']     = df['combined'].apply(len)
df['avg_word_len']   = df['combined'].apply(lambda x: np.mean([len(w) for w in str(x).split()]))
df['exclamation']    = df['combined'].apply(lambda x: str(x).count('!'))
df['question_marks'] = df['combined'].apply(lambda x: str(x).count('?'))
df['caps_ratio']     = df['combined'].apply(
    lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))

print('Cleaning and lemmatizing text (takes ~2 min on full dataset)...')
t0 = time.time()
df['clean_text'] = df['combined'].apply(preprocess)
print(f'Preprocessing complete in {time.time()-t0:.1f}s')
df[['combined', 'clean_text', 'word_count', 'caps_ratio', 'exclamation']].head()

In [ ]:
# ── CELL 4: EDA — Class Distribution & Subject Breakdown ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EDA — Dataset Overview', fontsize=15, fontweight='bold', y=1.02)

counts = df['label'].value_counts()
axes[0].pie([counts[0], counts[1]], labels=['Fake', 'Real'],
            colors=[FAKE_COLOR, REAL_COLOR], autopct='%1.1f%%', startangle=90,
            textprops={'color': '#e0e0ff'},
            wedgeprops={'edgecolor': '#0f0f1a', 'linewidth': 2})
axes[0].set_title('Class Distribution')

subj_f = fake['subject'].value_counts().head(6)
subj_r = real['subject'].value_counts().head(6)
x = np.arange(len(subj_f))
axes[1].bar(x-0.2, subj_f.values, 0.4, label='Fake', color=FAKE_COLOR, alpha=0.85)
axes[1].bar(x+0.2, subj_r.reindex(subj_f.index, fill_value=0).values,
            0.4, label='Real', color=REAL_COLOR, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(subj_f.index, rotation=30, ha='right', fontsize=9)
axes[1].set_title('Subject Distribution')
axes[1].legend()

bars = axes[2].bar(['Fake', 'Real'], [counts[0], counts[1]],
                   color=[FAKE_COLOR, REAL_COLOR], edgecolor='#0f0f1a', linewidth=2)
for bar, val in zip(bars, [counts[0], counts[1]]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{val:,}', ha='center', va='bottom')
axes[2].set_title('Article Counts')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_class_dist.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 5: EDA — Text Length Histograms ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EDA — Text Length & Style Features', fontsize=15, fontweight='bold')

for ax, feat, title in zip(axes,
    ['word_count', 'char_count', 'avg_word_len'],
    ['Word Count', 'Character Count', 'Avg Word Length']):
    cap = df[feat].quantile(0.98)
    ax.hist(df[df.label==0][feat].clip(upper=cap), bins=50,
            alpha=0.7, color=FAKE_COLOR, label='Fake', density=True)
    ax.hist(df[df.label==1][feat].clip(upper=cap), bins=50,
            alpha=0.7, color=REAL_COLOR, label='Real', density=True)
    ax.set_title(title)
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('eda_lengths.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 6: EDA — Word Clouds ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('EDA — Most Frequent Words', fontsize=15, fontweight='bold')

fake_text = ' '.join(df[df.label==0]['clean_text'])
real_text = ' '.join(df[df.label==1]['clean_text'])

wc_fake = WordCloud(width=800, height=400, background_color='#1a1a2e',
                    colormap='Reds', max_words=100).generate(fake_text)
wc_real = WordCloud(width=800, height=400, background_color='#1a1a2e',
                    colormap='Blues', max_words=100).generate(real_text)

axes[0].imshow(wc_fake, interpolation='bilinear')
axes[0].set_title('FAKE News — Top Words', color=FAKE_COLOR)
axes[0].axis('off')
axes[1].imshow(wc_real, interpolation='bilinear')
axes[1].set_title('REAL News — Top Words', color=REAL_COLOR)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('eda_wordclouds.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 7: EDA — Top N-Grams ─────────────────────────────────────────────────
def top_ngrams(corpus, n=2, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n), max_features=top_k)
    bag = vec.fit_transform(corpus)
    return sorted(zip(vec.get_feature_names_out(), bag.sum(axis=0).A1), key=lambda x: -x[1])

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('EDA — Top N-Grams: Fake vs Real', fontsize=15, fontweight='bold')

for row, n in enumerate([2, 3]):
    for col, (corpus, color, cname) in enumerate([
        (df[df.label==0]['clean_text'], FAKE_COLOR, 'Fake'),
        (df[df.label==1]['clean_text'], REAL_COLOR, 'Real')
    ]):
        grams = top_ngrams(corpus, n=n)
        words, counts = zip(*grams)
        ax = axes[row][col]
        ax.barh(words[::-1], counts[::-1], color=color, alpha=0.85, edgecolor='#0f0f1a')
        ax.set_title(f'{cname} — Top {"Bigrams" if n==2 else "Trigrams"}')
        ax.set_xlabel('Frequency')
        ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig('eda_ngrams.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 8: EDA — Stylometric Boxplots ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA — Stylometric Signals', fontsize=14, fontweight='bold')

sns.boxplot(x='label', y='caps_ratio', data=df,
            palette={0: FAKE_COLOR, 1: REAL_COLOR}, ax=axes[0])
axes[0].set_xticklabels(['Fake', 'Real'])
axes[0].set_title('Capital Letter Ratio')
axes[0].set_xlabel('')

df_clip = df.copy()
df_clip['exclamation'] = df_clip['exclamation'].clip(upper=20)
sns.boxplot(x='label', y='exclamation', data=df_clip,
            palette={0: FAKE_COLOR, 1: REAL_COLOR}, ax=axes[1])
axes[1].set_xticklabels(['Fake', 'Real'])
axes[1].set_title('Exclamation Mark Count (clipped @20)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('eda_stylometric.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 9: TF-IDF Vectorization & Train/Test Split ───────────────────────────
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF with unigrams + bigrams
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Train shape : {X_train_tfidf.shape}')
print(f'Test shape  : {X_test_tfidf.shape}')

In [ ]:
# ── CELL 10: Train All Baseline Models ────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, learning_rate=0.1,
                                         eval_metric='logloss', random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    y_pred  = model.predict(X_test_tfidf)
    elapsed = time.time() - t0
    y_score = (model.predict_proba(X_test_tfidf)[:, 1]
               if hasattr(model, 'predict_proba')
               else model.decision_function(X_test_tfidf))
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_score': y_score,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_score),
        'time_s':    elapsed,
    }
    print(f"{name:25s} Acc={results[name]['accuracy']:.4f}  "
          f"F1={results[name]['f1']:.4f}  AUC={results[name]['roc_auc']:.4f}  "
          f"{elapsed:.1f}s")

In [ ]:
# ── CELL 11: Hyperparameter Tuning — Logistic Regression ──────────────────────
print('GridSearchCV on Logistic Regression (5-fold CV)...')
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid={
        'C':       [0.01, 0.1, 1.0, 10.0],
        'penalty': ['l2'],
        'solver':  ['lbfgs', 'saga'],
        'max_iter':[1000],
    },
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train_tfidf, y_train)
print(f'Best LR params : {lr_grid.best_params_}')
print(f'Best CV F1     : {lr_grid.best_score_:.4f}')

In [ ]:
# ── CELL 12: Hyperparameter Tuning — XGBoost ──────────────────────────────────
print('GridSearchCV on XGBoost (3-fold CV)...')
xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_grid={
        'n_estimators':  [100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth':     [3, 5, 7],
    },
    cv=3, scoring='f1', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train_tfidf, y_train)
print(f'Best XGB params : {xgb_grid.best_params_}')
print(f'Best CV F1      : {xgb_grid.best_score_:.4f}')

# Register tuned models
for tag, grid in [('LR (Tuned)', lr_grid), ('XGB (Tuned)', xgb_grid)]:
    best    = grid.best_estimator_
    y_pred  = best.predict(X_test_tfidf)
    y_score = best.predict_proba(X_test_tfidf)[:, 1]
    results[tag] = {
        'model': best, 'y_pred': y_pred, 'y_score': y_score,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_score),
        'time_s':    0,
    }
    print(f"{tag:25s} F1={results[tag]['f1']:.4f}  AUC={results[tag]['roc_auc']:.4f}")

In [ ]:
# ── CELL 13: Metrics Summary Table ────────────────────────────────────────────
metrics_df = pd.DataFrame([
    {'Model': k, 'Accuracy': v['accuracy'], 'Precision': v['precision'],
     'Recall': v['recall'], 'F1-Score': v['f1'], 'ROC-AUC': v['roc_auc'],
     'Train Time (s)': round(v['time_s'], 1)}
    for k, v in results.items()
]).set_index('Model').sort_values('F1-Score', ascending=False)

print(metrics_df.round(4).to_string())
metrics_df.style.background_gradient(cmap='Greens', subset=['F1-Score','ROC-AUC'])

In [ ]:
# ── CELL 14: Model Comparison Dashboard ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Result Dashboard — Model Comparison', fontsize=15, fontweight='bold')

# Grouped bar chart
x, width = np.arange(len(metrics_df)), 0.16
palette = ['#4361ee', REAL_COLOR, '#f8961e', FAKE_COLOR, '#a8dadc']
for i, (metric, color) in enumerate(zip(
        ['Accuracy','Precision','Recall','F1-Score','ROC-AUC'], palette)):
    axes[0].bar(x + i*width, metrics_df[metric], width,
                label=metric, color=color, alpha=0.85)
axes[0].set_xticks(x + 2*width)
axes[0].set_xticklabels(metrics_df.index, rotation=25, ha='right', fontsize=8)
axes[0].set_ylim(0.85, 1.01)
axes[0].set_title('All Metrics per Model')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.2)

# ROC curves
for name, v in results.items():
    fpr, tpr, _ = roc_curve(y_test, v['y_score'])
    axes[1].plot(fpr, tpr, linewidth=2, label=f"{name} ({v['roc_auc']:.3f})")
axes[1].plot([0,1],[0,1],'--', color='gray', linewidth=1, label='Random')
axes[1].set_title('ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('dashboard_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 15: Confusion Matrices — Top 3 Models ────────────────────────────────
top3 = metrics_df.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices — Top 3 Models', fontsize=14, fontweight='bold')

for ax, name in zip(axes, top3):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='magma',
                xticklabels=['Fake','Real'], yticklabels=['Fake','Real'],
                ax=ax, linewidths=2, linecolor='#0f0f1a')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 16: 5-Fold Cross-Validation Stability ────────────────────────────────
print('Running 5-fold CV on top 3 models...')
cv_results = {}
for name in top3:
    scores = cross_val_score(
        results[name]['model'], X_train_tfidf, y_train, cv=5, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:25s}  F1: {scores.mean():.4f} ± {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f0f1a')
ax.boxplot(cv_results.values(), labels=cv_results.keys(),
           patch_artist=True,
           boxprops=dict(facecolor='#1a1a2e', color=REAL_COLOR),
           medianprops=dict(color=ACCENT, linewidth=2),
           whiskerprops=dict(color='#e0e0ff'),
           capprops=dict(color='#e0e0ff'),
           flierprops=dict(markerfacecolor=FAKE_COLOR, marker='o'))
ax.set_title('5-Fold CV F1 Distribution (Top 3 Models)')
ax.set_ylabel('F1 Score')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('cv_stability.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── CELL 17: Feature Importance (Coefficients / SHAP) ─────────────────────────
best_name     = metrics_df.index[0]
best_model    = results[best_name]['model']
feature_names = tfidf.get_feature_names_out()

if hasattr(best_model, 'coef_'):
    coef         = best_model.coef_[0]
    top_fake_idx = np.argsort(coef)[:20]
    top_real_idx = np.argsort(coef)[-20:][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')

    axes[0].barh(feature_names[top_fake_idx], coef[top_fake_idx],
                 color=FAKE_COLOR, alpha=0.85)
    axes[0].set_title('Top Fake Indicators', color=FAKE_COLOR)
    axes[0].invert_xaxis()
    axes[0].grid(axis='x', alpha=0.2)

    axes[1].barh(feature_names[top_real_idx], coef[top_real_idx],
                 color=REAL_COLOR, alpha=0.85)
    axes[1].set_title('Top Real Indicators', color=REAL_COLOR)
    axes[1].grid(axis='x', alpha=0.2)

    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()

else:
    idx          = np.random.choice(X_test_tfidf.shape[0], 500, replace=False)
    X_sample     = X_test_tfidf[idx]
    explainer    = shap.TreeExplainer(best_model)
    shap_values  = explainer.shap_values(X_sample)
    plt.figure(figsize=(12, 8), facecolor='#0f0f1a')
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      max_display=20, show=False, plot_type='bar')
    plt.title(f'SHAP Feature Importance — {best_name}')
    plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()

In [ ]:
# ── CELL 18: Full Classification Reports ──────────────────────────────────────
for name in top3:
    print(f'\n{'='*50}')
    print(f'  {name}')
    print('='*50)
    print(classification_report(y_test, results[name]['y_pred'],
                                target_names=['Fake', 'Real']))

In [ ]:
# ── CELL 19: Live Inference ────────────────────────────────────────────────────
def predict_news(text, model=best_model, vectorizer=tfidf):
    cleaned = preprocess(text)
    vec     = vectorizer.transform([cleaned])
    pred    = model.predict(vec)[0]
    label   = 'REAL' if pred == 1 else 'FAKE'
    if hasattr(model, 'predict_proba'):
        conf = max(model.predict_proba(vec)[0])
        print(f'[{label}]  confidence: {conf:.2%}')
    else:
        print(f'[{label}]')
    return pred

# Test articles
predict_news('Federal Reserve holds interest rates steady, citing concerns about inflation outlook.')
predict_news('EXPOSED!! Deep state elites hiding alien contact — mainstream media covering up the TRUTH!!')